In [1]:
import numpy as np
import pandas as pd
import itertools

In [2]:
def evaluate_models(models, user_selected, data, disjunction=False):
    n,m = data.shape
    k = len(models.columns)
    if disjunction:
        model_selected = pd.DataFrame((models.values[None]*data.values[:,:,None]).sum(axis=1), columns=models.columns)
    else:
        model_selected = pd.DataFrame(((models.values[None]*data.values[:,:,None]) + (1-models.values[None])).prod(axis=1), columns=models.columns)

    tp = pd.DataFrame(model_selected*user_selected.values[:,None], columns=models.columns)
    fp = pd.DataFrame(model_selected*(1-user_selected.values[:,None]), columns=models.columns)
    tn = pd.DataFrame((1-model_selected)*(1-user_selected.values[:,None]), columns=models.columns)
    fn = pd.DataFrame((1-model_selected)*user_selected.values[:,None], columns=models.columns)

    sensitivity_loss = 1 - (tp.sum().sum() / (tp.sum().sum()+fn.sum().sum())) # 1 - sensitivity (true positive rate)
    specificity_loss = 1 - tn.sum().sum() / (tn.sum().sum()+fp.sum().sum()) # 1 - specificity (true negative rate)
    if disjunction:
        diff = models.diff()
        num_neg = (diff[diff==-1].sum().abs().sum()).sum() / (m*k)
        num_pos = (diff[diff==1].sum().abs().sum()).sum() / (m*k)
        sparsity_loss = min(num_neg, num_pos) # counts the number of continuous segments for binned numeric features
    else:
        sparsity_loss = (models.sum()-1).sum() / (m*k) # counts the number of active binary features
    feature_coverage_loss = 1 - (models.sum(axis=1)>0).sum()/m # counts the number of features that are active for any predicate
    
    return model_selected, tp, fp, tn, fn, sensitivity_loss, specificity_loss, sparsity_loss, feature_coverage_loss

In [3]:
def evaluate_all_models(user_selected, data, k, disjunction=False):
    n,m = data.shape
    all_models_ = list(itertools.product(list(itertools.product(*[[0, 1] for l in range(k)])), repeat=m))
    loss_list = []
    all_model_selected = []
    all_models = []
    all_tp = []
    all_fp = []
    all_tn = []
    all_fn = []
    for models_ in all_models_:
        models = pd.DataFrame(models_, columns=[f'predicate_{l}' for l in range(k)], index=data.columns)
        res = evaluate_models(models, user_selected, data, disjunction)
        model_selected, tp, fp, tn, fn, sensitivity_loss, specificity_loss, sparsity_loss, feature_coverage_loss = res
        loss_list.append(res[5:])
        all_model_selected.append(res[0])
        all_models.append(models)
        all_tp.append(res[1])
        all_fp.append(res[2])
        all_tn.append(res[3])
        all_fn.append(res[4])
    loss = pd.DataFrame(loss_list, columns=['sensitivity', 'specificity', 'sparsity', 'feature_coverage'])
    if disjunction:
        loss['total'] = loss[['sensitivity', 'specificity', 'sparsity']].sum(axis=1) # ignoring feature_coverage loss for binned numeric features for now
    else:
        loss['total'] = loss.sum(axis=1) # sensitivity + specificity + sparsity + feature_coverage
    loss['accuracy'] = loss['sensitivity'] + loss['specificity']
    return all_models, loss

In [4]:
n = 1000
m = 5

# Binary Features

In [5]:
binary_data = pd.DataFrame(np.random.binomial(1, .5, size=(n, m)))
binary_data.columns = [f'binary_{j}' for j in range(m)]
binary_data['binary_1'] = binary_data['binary_0']

## Ground Truth

**("binary_0" = 1) OR ("binary_1" = 1)**

In [6]:
user_selected_binary1 = binary_data['binary_0']

### Learn 1 predicate

In [7]:
predicates1a, loss1a = evaluate_all_models(user_selected_binary1, binary_data, 1)

In [8]:
loss1a.sort_values('total').head()

,sensitivity,specificity,sparsity,feature_coverage,total,accuracy
16,0.000000,0.0,0.0,0.8,0.800000,0.000000
24,0.000000,0.0,0.2,0.6,0.800000,0.000000
8,0.000000,0.0,0.0,0.8,0.800000,0.000000
18,0.487854,0.0,0.2,0.6,1.287854,0.487854
10,0.487854,0.0,0.2,0.6,1.287854,0.487854


*Either binary_0 or binary_1: lower sparsity loss, higher feature-coverage loss*

In [9]:
predicates1a[8]

,predicate_0
binary_0,0
binary_1,1
binary_2,0
binary_3,0
binary_4,0


In [10]:
predicates1a[16]

,predicate_0
binary_0,1
binary_1,0
binary_2,0
binary_3,0
binary_4,0


*binary_0 and binary_1: higher sparsity loss, lower feature-coverage loss*

In [11]:
predicates1a[24]

,predicate_0
binary_0,1
binary_1,1
binary_2,0
binary_3,0
binary_4,0


### Learn 2 predicates

In [12]:
predicates1b, loss1b = evaluate_all_models(user_selected_binary1, binary_data, 2)

In [13]:
loss1b.sort_values('total').head()

,sensitivity,specificity,sparsity,feature_coverage,total,accuracy
576,0.0,0.0,0.0,0.6,0.6,0.0
384,0.0,0.0,0.0,0.6,0.6,0.0
704,0.0,0.0,0.1,0.6,0.7,0.0
832,0.0,0.0,0.1,0.6,0.7,0.0
896,0.0,0.0,0.1,0.6,0.7,0.0


In [14]:
predicates1b[384]

,predicate_0,predicate_1
binary_0,0,1
binary_1,1,0
binary_2,0,0
binary_3,0,0
binary_4,0,0


## Ground Truth

**(("binary_0" = 1) AND ("binary_2" = 1)) OR (("binary_1" = 1) AND ("binary_2" = 1))**

In [15]:
user_selected_binary2 = binary_data['binary_0']*binary_data['binary_2']

### Learn 2 predicates

In [16]:
predicates2, loss2 = evaluate_all_models(user_selected_binary2, binary_data, 2)

In [17]:
loss2.sort_values('total').head()

,sensitivity,specificity,sparsity,feature_coverage,total,accuracy
432,0.0,0.000000,0.2,0.4,0.600000,0.000000
624,0.0,0.000000,0.2,0.4,0.600000,0.000000
592,0.0,0.161765,0.1,0.4,0.661765,0.161765
608,0.0,0.161765,0.1,0.4,0.661765,0.161765
400,0.0,0.161765,0.1,0.4,0.661765,0.161765


In [18]:
predicates2[624]

,predicate_0,predicate_1
binary_0,1,0
binary_1,0,1
binary_2,1,1
binary_3,0,0
binary_4,0,0


# Numeric Features

In [19]:
numeric_data = pd.DataFrame(np.random.uniform(size=(n, m)))
numeric_data.columns = [f'numeric_{j}' for j in range(m)]
numeric_data['numeric_1'] = numeric_data['numeric_0']

## Ground Truth

**("numeric_0" > .5)**

In [20]:
num_bins = 10
numeric_data_binned = {col: pd.get_dummies(pd.cut(numeric_data[col], bins=num_bins)) for col in numeric_data.columns}

In [21]:
user_selected_numeric1 = (numeric_data['numeric_0']>.5).astype(int)

In [22]:
predicates3, loss3 = evaluate_all_models(user_selected_numeric1, numeric_data_binned['numeric_0'], 1, disjunction=True)

In [23]:
loss3.sort_values('total').head()

,sensitivity,specificity,sparsity,feature_coverage,total,accuracy
31,0.000000,0.000000,0.0,0.5,0.000000,0.000000
15,0.184676,0.000000,0.0,0.6,0.184676,0.184676
63,0.000000,0.205703,0.0,0.4,0.205703,0.205703
287,0.000000,0.173116,0.1,0.4,0.273116,0.173116
29,0.188605,0.000000,0.1,0.6,0.288605,0.188605


In [24]:
predicates3[31]

,predicate_0
"(0.000266, 0.101]",0
"(0.101, 0.201]",0
"(0.201, 0.301]",0
"(0.301, 0.401]",0
"(0.401, 0.501]",0
"(0.501, 0.6]",1
"(0.6, 0.7]",1
"(0.7, 0.8]",1
"(0.8, 0.9]",1
"(0.9, 1.0]",1


## Ground Truth

**(("numeric_0" > .66) OR ("numeric_0" < .33)) AND (("numeric_0" <= .101) OR ("numeric_0" > .201))**

In [29]:
user_selected_numeric2 = (
    ((numeric_data['numeric_0']>.66) | (numeric_data['numeric_0']<.33)) &
    ((numeric_data['numeric_0']<=.101) | (numeric_data['numeric_0']>.201))
).astype(int)

In [30]:
predicates4, loss4 = evaluate_all_models(user_selected_numeric2, numeric_data_binned['numeric_0'], 1, disjunction=True)

In [31]:
loss4.sort_values('total').head()

,sensitivity,specificity,sparsity,feature_coverage,total,accuracy
647,0.127367,0.000000,0.2,0.5,0.327367,0.127367
519,0.297762,0.000000,0.1,0.6,0.397762,0.297762
135,0.309811,0.000000,0.1,0.6,0.409811,0.309811
711,0.068847,0.157518,0.2,0.4,0.426365,0.226365
903,0.127367,0.202864,0.1,0.4,0.430231,0.330231


In [32]:
predicates4[647]

,predicate_0
"(0.000266, 0.101]",1
"(0.101, 0.201]",0
"(0.201, 0.301]",1
"(0.301, 0.401]",0
"(0.401, 0.501]",0
"(0.501, 0.6]",0
"(0.6, 0.7]",0
"(0.7, 0.8]",1
"(0.8, 0.9]",1
"(0.9, 1.0]",1
